# Nested Loops and Common Patterns

**Topic:** `2.4` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*Nested loops multiply the work: when they are the right tool, when a sliding window or a closed form replaces them, and why almost every bug in one is a boundary error.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

## 1. Why This Topic Matters

Grid mapping, coverage surveys and neighbour planning are all nested loops over real sensor data that is rarely the tidy shape a textbook draws. A loop that assumes a fixed width crashes the moment a row comes back short.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Compute the cost of nested loops as a product, and count the passes to prove it.
2. Iterate rows rather than a hard-coded inner bound so ragged data is safe.
3. Explain why the inner loop restarts on every outer pass.
4. Replace a nested window scan with a sliding window in one pass.
5. Recognise a closed form, such as the square of the sum for a pairwise product.
6. Return from a nested search to stop both loops at once.

## 3. Prerequisites

This topic assumes you already have:

- Topic 2.3 Loop Control

## 4. Mental Model

**Mental model: a clock, not a matrix.** The outer loop is the slow hand and the inner loop is the fast one. The fast hand sweeps all the way round once for every tick of the slow hand, and it never remembers where it was - it always restarts from the beginning.

|  | Outer loop | Inner loop |
| --- | --- | --- |
| Restarts | once | on every outer pass |
| Cost | n iterations | m per outer pass, so n*m total |
| Knows its index | only if enumerated | only if enumerated |
| Best replaced by | a single pass | a slice or a derived value |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Nested loop | A loop inside another loop. |
| Iteration count | The product of the loop lengths. |
| Ragged grid | Rows of differing lengths, which break hard-coded bounds. |
| enumerate | Iterate a sequence while also yielding its index. |
| Sliding window | Each window of fixed size along a sequence, in one pass. |
| Off-by-one | A bound error that visits one element too few or too many. |
| Closed form | An algebraic expression replacing an enumeration. |
| Early exit | Returning from inside a nested search to stop the outer loop too. |
| Cartesian product | Every ordered pair drawn from two sequences. |
| Pairwise sum | The sum of all products of pairs, equal to the square of the sum. |

## 6. Conceptual Explanation

A loop inside a loop runs its body once for every pass of the outer loop. That single fact explains both the usefulness and the danger: nested loops are how you visit every cell of a grid or every pair of items, and they are also how a program that should do one hundred operations quietly does ten thousand.

In [ ]:
grid = ((0, 1, 0),
        (0, 0, 1),
        (1, 0, 0))

rows = cols = 0
for row in grid:
    rows += 1
    for cell in row:
        cols += 1
print(f'outer ran {rows} times, inner ran {cols} times')

The cost multiplies. An outer loop of *n* iterations containing an inner loop of *m* iterations performs *n × m* steps. With three passes each that is nine steps; add a third level and it becomes a cubic cost, which is how a routine that handles a small table becomes unusable on a full sensor grid.

In [ ]:
def steps_two_levels(n, m) -> int:
    steps = 0
    for _ in range(n):
        for _ in range(m):
            steps += 1
    return steps


def steps_three_levels(n, m, k) -> int:
    steps = 0
    for _ in range(n):
        for _ in range(m):
            for _ in range(k):
                steps += 1
    return steps


print('two levels 100:', steps_two_levels(100, 100))
print('three levels 20:', steps_three_levels(20, 20, 20))

> **Early exit cuts the work**
>
> A `break` in the inner loop does not reduce the outer count, but a `return` from inside a nested search removes the remaining outer iterations entirely - which is why search helpers return rather than set a flag.

## 7. Formal Theory

Nested iteration composes. The outer loop produces an element, and the inner loop produces one element for each of *its* sequence - so the body of the inner loop runs once per ordered **pair**. That is the right way to think about it: the inner loop restarts from the beginning on every pass of the outer one.

```text
for a in A:  for b in B:  ...   ==   visit every (a, b) pair, |A| * |B| of them
```

The classical patterns that matter here each reduce this to a different shape. Diagonal iteration adds an offset so the two indices move together. Sliding windows slice a fixed-size window along a sequence. A running comparison carries one value from the previous pass. Recognising which pattern a problem wants is the real skill - it is what separates a cubic loop from a linear pass.

In [ ]:
def diagonal(size: int) -> list[tuple]:
    return [(r, c) for r in range(size) for c in range(r, size)]


def window(values, size: int) -> list[tuple]:
    return [
        tuple(values[i:i + size])
        for i in range(len(values) - size + 1)
    ]


print('diagonal(3):', diagonal(3))
print('window(4, 2):', window((1, 2, 3, 4), 2))

| Pattern | Shape | Cost |
| --- | --- | --- |
| Full nested scan | every (a, b) pair | O(n * m) |
| Early-exit search | until the first match | O(n * m) worst, often far less |
| Sliding window | each window of size k | O(n) |
| Carried value | one value from the previous pass | O(n) |

## 8. Syntax

The canonical nested scan enumerates both levels, so the reader can see the coordinates rather than inferring them.

In [ ]:
GRID = ((0, 1, 0),
        (0, 0, 1),
        (1, 0, 0))

for row_index, row in enumerate(GRID):
    for col_index, cell in enumerate(row):
        if cell:
            print(f'occupied at ({row_index}, {col_index})')

The anti-pattern - a hard-coded bound that quietly breaks on short rows:

```text
grid = ((0, 1),
        (0, 0, 1))     # ragged: the second row is longer
for row_index in range(3):
    for col_index in range(3):   # IndexError on a smaller grid
        print(grid[row_index][col_index])

for row_index, row in enumerate(grid):
    for col_index, cell in enumerate(row):   # safe for any shape
        print(row_index, col_index, cell)
```

> **Never index a nested loop by a hard-coded bound**
>
> Iterate the rows themselves. A hard-coded column count raises IndexError on a smaller grid and silently ignores extra cells on a larger one.

## 9. Basic Examples

**Example 1 - counting with a nested loop.**

In [ ]:
pairs = ('a', 'b')
count = 0
for first in pairs:
    for second in pairs:
        count += 1
print(f'{len(pairs)} * {len(pairs)} = {count} pairs')

**Example 2 - a linear alternative to a nested sum.**

In [ ]:
values = [1, 2, 3, 4]

pairs_sum = 0
for a in values:
    for b in values:
        pairs_sum += a * b

flat_sum = sum(values)
print('nested :', pairs_sum)
print('flat   :', flat_sum * flat_sum)
print('equal  :', pairs_sum == flat_sum * flat_sum)

## 10. Intermediate Examples

Scanning an occupancy grid is the canonical nested search. The `return` on the first hit is what keeps it from walking the whole grid once a match is found.

In [ ]:
GRID = ((0, 0, 1),
        (0, 1, 0),
        (0, 0, 0))


def first_obstacle(grid):
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            if cell:
                return row_index, col_index
    return None


print('first obstacle:', first_obstacle(GRID))
print('clear grid    :', first_obstacle(((0, 0), (0, 0))))

## 11. Advanced Examples

Some nested-loop problems are not really nested. A pairwise sum over a list is the square of the sum, and a sliding window is one pass with a slice. Knowing which is which turns an O(n^2) routine into an O(n) one.

In [ ]:
values = (1, 2, 3, 4)


def every_pair_sum(values):
    """O(n^2): the direct translation of 'for each pair'."""
    total = 0
    for a in values:
        for b in values:
            total += a * b
    return total


def every_pair_sum_fast(values):
    """O(n): the same number, derived rather than enumerated."""
    return sum(values) ** 2


print('direct:', every_pair_sum(values))
print('fast  :', every_pair_sum_fast(values))

The two are equal because expanding `(a1 + a2 + ...)^2` produces exactly the cross terms the double loop accumulates. Recognising that identity is worth more than any micro-optimisation.

## 12. Code Walkthrough

Below is a coverage checker for a survey grid. It walks every cell, applies a predicate, and reports what was and was not covered - with an explicit guard so a ragged grid cannot raise.

In [ ]:
"""coverage.py - check a survey grid against a minimum sampling rule."""
from __future__ import annotations

MIN_COVERAGE = 0.5


def check_coverage(grid, is_covered, min_coverage=MIN_COVERAGE) -> dict:
    """Return coverage statistics for a grid of any rectangular shape.

    Iterating the rows rather than a fixed width means a ragged or
    empty grid is measured as it actually is, not assumed away.
    """
    total = 0
    covered = 0
    uncovered: list[tuple] = []
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            total += 1
            if is_covered(cell):
                covered += 1
            else:
                uncovered.append((row_index, col_index))
    ratio = (covered / total) if total else 0.0
    return {
        'total': total,
        'covered': covered,
        'ratio': round(ratio, 3),
        'ok': ratio >= min_coverage,
        'uncovered': uncovered,
    }

The `total` counter is what makes the empty-grid case honest: dividing by a zero cell count would raise, and silently returning 1.0 would claim a full survey of nothing. Reporting `uncovered` coordinates means the caller learns *where* to send the robot, not merely that the grid failed.

## 13. Line-by-Line Explanation

1. MIN_COVERAGE is a module constant so the survey policy is visible above the loop rather than buried in a comparison.
2. Iterating each row rather than a fixed width means the function works for ragged and empty grids without a guard at every level.
3. total is incremented before any classification, so the denominator counts every cell that exists.
4. The else-style branch collects uncovered coordinates rather than just counting them, which tells the caller where to send the robot.
5. The conditional expression handles the empty grid explicitly instead of letting a division by zero escape.
6. ok is derived from the ratio, so the verdict cannot disagree with the measurement it came from.

## 14. Common Mistakes

**Mistake 1 - a hard-coded inner bound.**

```text
grid = ((0, 1),
        (0, 0, 1))          # ragged
for r in range(3):
    for c in range(3):
        print(grid[r][c])      # IndexError on a short row

for r, row in enumerate(grid):
    for c, cell in enumerate(row):
        print(r, c, cell)     # safe for any shape
```

**Mistake 2 - assuming the inner loop restarts the outer one.**

```text
total = 0
for a in (1, 2):
    for b in (1, 2, 3):
        total += 1
print(total)          # 6, not 3: the inner loop restarts each pass
```

**Mistake 3 - an off-by-one that drops the last row or column.**

```text
for r in range(len(grid) - 1):        # drops the final row
    for c in range(len(grid[0]) - 1): # drops the final column
        ...

for r, row in enumerate(grid):        # visits every cell
    for c, cell in enumerate(row):
        ...
```

**Mistake 4 - a third level where a second would do.**

```text
for a in A:
    for b in B:
        for c in C:      # O(n^3) where O(n^2) suffices
            ...
```

## 15. Debugging Techniques

For a nested loop, print the pair on each pass. Seeing every (row, col) the body visits turns a suspected off-by-one into a visible list.

In [ ]:
grid = ((0, 1, 0), (0, 0))
visits = []
for r, row in enumerate(grid):
    for c, cell in enumerate(row):
        visits.append((r, c))
print('visited:', visits)
print('expected:', [(r, c) for r in range(2) for c in (2, 3)])

For a suspected cost problem, count the passes rather than timing them. If the pass count is far larger than the data would suggest, the bound is wrong before the performance is.

In [ ]:
def count_steps_two(n, m) -> int:
    steps = 0
    for _ in range(n):
        for _ in range(m):
            steps += 1
    return steps


for n in (10, 100, 1000):
    steps = count_steps_two(n, n)
    print(f'n={n:<5} steps={steps:<9} ratio_vs_n={steps / n:.0f}')

> **Compare against the derived value**
>
> For an n by n grid the step count should be exactly n squared. Any other number means the loop is not doing what the problem statement asked.

## 16. Best Practices

- Iterate the rows themselves; never hard-code an inner bound.
- Use enumerate at both levels so coordinates are visible.
- Return from a nested search rather than setting a flag.
- Check whether a nested loop is really needed before writing one.
- Derive a closed form when the pair sum or count is algebraically known.
- Test with a ragged grid as well as a rectangular one.
- Count steps first, then time them, when cost is in doubt.
- Prefer a sliding window over repeated slicing inside a loop.

## 17. Performance Considerations

Nested loops multiply cost, and the multiplier is easy to underestimate. Three levels of a thousand elements each is a billion steps - not a slow program, an unfinished one. Before writing the second loop, ask what the body actually has to do, and whether the same result is available from a single pass.

Two transformations remove most nested loops in practice. A sliding window replaces a nested scan of every k-sized window with one pass and a slice per position, turning O(n*k) into O(n). A closed form replaces enumeration with algebra: the sum of all pairwise products is the square of the sum, and a count of pairs above a threshold can often be computed from sorted data in one pass.

> **Measure the step count, not the clock**
>
> Instrumenting a counter tells you the shape of the cost immediately and without the noise of a real machine. If the count is right, the time usually is too.

## 18. Pythonic Approaches

A nested pair scan against its linear equivalent.

In [ ]:
values = (1, 2, 3, 4)

# before: enumerate every ordered pair
pairs = [(a, b) for a in values for b in values]
print('pair count:', len(pairs))

# after: the count is derived, not enumerated
print('derived   :', len(values) ** 2)

# and the product sum is the square of the total
print('sum of products:', sum(values) ** 2)

## 19. Robotics Connection

A coverage grid is the shape almost every mapping task takes: nested loops over rows and cells, with an early exit when a cell already answers the question.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

GRID = ((0, 0, 1), (0, 1, 0), (0, 0, 0))
visited = 0
for row_index, row in enumerate(GRID):
    for col_index, cell in enumerate(row):
        visited += 1
        if cell:
            print(f'obstacle at ({row_index}, {col_index})')
print(f'visited {visited} cells, battery {state["battery_pct"]:.1f}%')

## 20. Engineering Example

A nested neighbour scan returning the strongest match rather than every match - the common shape of a local planner step.

In [ ]:
def strongest_neighbour(grid, row, col, default=None):
    """Return the (dr, dc) step toward the largest neighbouring cell."""
    best = None
    best_value = default
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue
            r, c = row + dr, col + dc
            if not (0 <= r < len(grid) and 0 <= c < len(grid[r])):
                continue
            if grid[r][c] > best_value:
                best_value = grid[r][c]
                best = (dr, dc)
    return best

## 21. Guided Practice

1. Count the iterations of a 3 by 4 nested loop and confirm it is 12.
2. Write a nested scan that prints every coordinate of a ragged grid.
3. Write a sliding window of size 2 over a five-element list.
4. Write a nested search returning the first (row, col) matching a value.

In [ ]:
grid = ((0, 1, 0), (0, 0))

count = 0
for r, row in enumerate(grid):
    for c, cell in enumerate(row):
        count += 1
print('cells visited:', count)

values = (1, 2, 3, 4, 5)
for start in range(len(values) - 1):
    print('window:', values[start:start + 2])

def find(grid, target):
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == target:
                return (r, c)
    return None

print('find 1 ->', find(grid, 1))
print('find 9 ->', find(grid, 9))

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Why does the inner loop appear to 'start over' on every outer pass?
- **Q1.** What is the total cost of n outer iterations containing m inner ones?
- **Q1.** The sum of `a * b` over every ordered pair equals what?
- **Q1.** Why does a rover's nested grid search return rather than break?

## 23. Summary

A nested loop runs its body once for every pass of the outer loop, so the work multiplies: *n* outer iterations with *m* inner ones perform *n × m* steps. That single fact is both why nested loops are the natural way to visit every cell of a grid or every pair of items, and why an extra level turns a routine that handles a small table into one that never finishes.

Two properties cause most of the bugs. The inner loop **restarts** on every outer pass, so it never continues where it left off - which surprises people who expect a matrix traversal. And a hard-coded inner bound such as `range(3)` raises on a smaller grid and silently ignores extra cells on a larger one. Iterating the rows themselves removes both problems, and a ragged grid becomes an ordinary input rather than a crash.

The most valuable skill here is recognising when a nested loop is not needed. A sliding window turns every k-sized window into one pass with a slice. A closed form replaces enumeration with algebra - the sum of all pairwise products is exactly the square of the sum, which takes an O(n²) loop and replaces it with one addition and one multiplication. And for searches, returning from inside the nested loop removes the remaining outer iterations entirely, which a `break` alone does not do.

## 24. Key Takeaways

- Nested loops multiply: n outer by m inner is n*m steps.
- The inner loop restarts on every outer pass.
- Never hard-code an inner bound; iterate the rows themselves.
- Use enumerate at both levels so coordinates are visible.
- Return from a nested search to stop both loops at once.
- A sliding window replaces a nested window scan with one pass.
- The pairwise product sum equals the square of the sum.
- Test with a ragged grid, not only a rectangular one.

## 25. Further Exploration

- [The for statement](https://docs.python.org/3/reference/compound_stmts.html#the-for-statement)
- [Nested comprehensions](https://docs.python.org/3/tutorial/datastructures.html#nested-list-comprehensions)
- [itertools.product - the cartesian product without a nested loop](https://docs.python.org/3/library/itertools.html#itertools.product)
- [collections.Counter for grid statistics](https://docs.python.org/3/library/collections.html#collections.Counter)